In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MaxNLocator

import json
import pickle
import torch

from scipy.stats import wilcoxon

# Compare best base models with FoMo-0D

24.10. model_24_10: SquareGMM of Simon, feat=70, inflate_full=True, delta=0.005, max_feature_dim=70, max_model_dim=70, prior.gmm.max_num_cluster=70, EPOCHS=50, BATCH_SIZE=6, nlayer=6, nhead=2,

---> BEST MODEL: model_24_10

In [7]:
#Fomo-0D model
default_dir = "../results/adbench/context5000.feat100.R500.inf-fullFalse.LTFalse.gen1tr1True.reuse100.E200.step1000.bs8.lr0.001.emb256.hdim512.nhead4.nlayer4.ndevice1/train_seed=0.transform=quantile.feat_truncation=subsampling"
default_dir_best_loss = "../ckpt/context5000.feat100.R500.inf-fullFalse.LTFalse.gen1tr1True.reuse100.E200.step1000.bs8.lr0.001.emb256.hdim512.nhead4.nlayer4.ndevice1/seed0/best.ckpt"

model_24_10_dir = "../results/adbench/(BESTMODEL)dataSGMM.context5000.feat70.R500.inf-fullTrue.LTFalse.gen1tr1True.reuse100.E50.step1000.bs6.lr0.001.emb256.hdim512.nhead2.nlayer6.ndevice1.numclusters70/train_seed=0.transform=quantile.feat_truncation=subsampling"

model_24_10_train_val_loss_dir = "../ckpt/(BESTMODEL)dataSGMM.context5000.feat70.R500.inf-fullTrue.LTFalse.gen1tr1True.reuse100.E50.step1000.bs6.lr0.001.emb256.hdim512.nhead2.nlayer6.ndevice1.numclusters70/seed0/train_val_loss.pickle"
model_24_10_best_dir = "../ckpt/(BESTMODEL)dataSGMM.context5000.feat70.R500.inf-fullTrue.LTFalse.gen1tr1True.reuse100.E50.step1000.bs6.lr0.001.emb256.hdim512.nhead2.nlayer6.ndevice1.numclusters70/seed0/best.ckpt"
model_24_10_best_val_dir = "../ckpt/(BESTMODEL)dataSGMM.context5000.feat70.R500.inf-fullTrue.LTFalse.gen1tr1True.reuse100.E50.step1000.bs6.lr0.001.emb256.hdim512.nhead2.nlayer6.ndevice1.numclusters70/seed0/best_val.ckpt"
model_24_10_last_dir = "../ckpt/(BESTMODEL)dataSGMM.context5000.feat70.R500.inf-fullTrue.LTFalse.gen1tr1True.reuse100.E50.step1000.bs6.lr0.001.emb256.hdim512.nhead2.nlayer6.ndevice1.numclusters70/seed0/last.ckpt"



### 1. Default model

In [4]:
default_aucroc_scores = []
default_aucpr_scores = []
default_f1_scores = []
datasets_name = []
# walk through all folders and find result.csv files
for subdir, _, files in os.walk(default_dir):
    if "result.csv" in files:
        file_path = os.path.join(subdir, "result.csv")
        df = pd.read_csv(file_path)
        # extract dataset name
        dataset_name = os.path.basename(os.path.dirname(subdir))
        datasets_name.append(dataset_name)

        # extract scores
        aucroc = df['aucroc'].values[0]
        default_aucroc_scores.append(aucroc)

        aucpr = df['aucpr'].values[0]
        default_aucpr_scores.append(aucpr)

        f1 = df['f1'].values[0]
        default_f1_scores.append(f1)

# compute average scores of all datasets
default_avg_aucroc = sum(default_aucroc_scores) / len(default_aucroc_scores)
default_avg_aucpr = sum(default_aucpr_scores) / len(default_aucpr_scores)
default_avg_f1 = sum(default_f1_scores) / len(default_f1_scores)

print(f"number of datasets: {len(default_aucroc_scores)}")
print(f"Average aucroc: {default_avg_aucroc}")
print(f"Average aucpr: {default_avg_aucpr}")
print(f"Average f1: {default_avg_f1}")

number of datasets: 121
Average aucroc: 0.7590645511541744
Average aucpr: 0.4786638498057685
Average f1: 0.4666975187169514


In [ ]:
default_fomo_dict = {'dataset': datasets_name, 'aucroc': default_aucroc_scores, 'aucpr': default_aucpr_scores, 'f1': default_f1_scores}
default_fomo_df = pd.DataFrame(default_fomo_dict)
#default_fomo_df.to_csv('default_fomo_result.csv')

In [28]:
default_ckpt_best = torch.load(default_dir_best_loss, map_location="cpu")

default_train_loss = default_ckpt_best["train_losses"]
default_val_loss = default_ckpt_best["val_losses"]
default_last_lr = default_ckpt_best["lr_schedulers"][0]['_last_lr']

print(default_ckpt_best.keys())
print(default_ckpt_best["epoch"])


dict_keys(['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'callbacks', 'optimizer_states', 'lr_schedulers', 'train_losses', 'val_losses'])
176


### 2. model_24_10 (best model)

In [9]:
model_24_10_aucroc_scores = []
model_24_10_aucpr_scores = []
model_24_10_f1_scores = []
datasets_name = []
# walk through all folders and find result.csv files
for subdir, _, files in os.walk(model_24_10_dir):
    if "result.csv" in files:
        file_path = os.path.join(subdir, "result.csv")
        df = pd.read_csv(file_path)
        # extract dataset name
        dataset_name = os.path.basename(os.path.dirname(subdir))
        datasets_name.append(dataset_name)
        
        # extract scores
        aucroc = df['aucroc'].values[0]
        model_24_10_aucroc_scores.append(aucroc)

        aucpr = df['aucpr'].values[0]
        model_24_10_aucpr_scores.append(aucpr)

        f1 = df['f1'].values[0]
        model_24_10_f1_scores.append(f1)

# compute average scores of all datasets
model_24_10_avg_aucroc = sum(model_24_10_aucroc_scores) / len(model_24_10_aucroc_scores)
model_24_10_avg_aucpr = sum(model_24_10_aucpr_scores) / len(model_24_10_aucpr_scores)
model_24_10_avg_f1 = sum(model_24_10_f1_scores) / len(model_24_10_f1_scores)

print(f"number of datasets: {len(model_24_10_aucroc_scores)}")
print(f"Average aucroc: {model_24_10_avg_aucroc}")
print(f"Average aucpr: {model_24_10_avg_aucpr}")
print(f"Average f1: {model_24_10_avg_f1}")

number of datasets: 121
Average aucroc: 0.7761609876983426
Average aucpr: 0.4592854784870678
Average f1: 0.46209364279753334


In [11]:
model_24_10_dict = {'dataset': datasets_name, 'aucroc': model_24_10_aucroc_scores, 'aucpr': model_24_10_aucpr_scores, 'f1': model_24_10_f1_scores}
model_24_10_df = pd.DataFrame(model_24_10_dict)
#model_24_10_df.to_csv('model_24_10_result.csv')

In [12]:
model_24_10_df

,dataset,aucroc,aucpr,f1
0,23_mammography,0.779666,0.319081,0.396154
1,SVHN_5,0.672675,0.183049,0.229282
2,40_vowels,0.703954,0.104582,0.060000
3,MVTec-AD_tile,0.971337,0.959816,0.868644
4,26_optdigits,0.918258,0.280503,0.260000
...,...,...,...,...
116,CIFAR10_0,0.695703,0.223519,0.269962
117,SVHN_9,0.543195,0.120002,0.159184
118,CIFAR10_5,0.488569,0.089251,0.053232
119,45_wine,1.000000,1.000000,1.000000


### P-value of default_fomo and model_24_10(best_model)

In [31]:
model_24_10_result = pd.read_csv('model_24_10_result.csv')
default_fomo_result = pd.read_csv('default_fomo_result.csv')

In [32]:
# better=[]
# model_24_10_auc, default_fomo_auc = [], []
# for ds, auc, default in zip(model_24_10_result["dataset"],model_24_10_result["aucroc"], default_fomo_result['aucroc']):

#     print(ds, auc, default)
#     better.append(auc >= default)
#     model_24_10_auc.append(auc)
#     default_fomo_auc.append(default)
# np.mean(better)

# from scipy.stats import wilcoxon

# test = wilcoxon(model_24_10_auc, default_fomo_auc)
# print(test)

In [33]:
import numpy as np
from scipy.stats import wilcoxon
model_24_10_auc, default_fomo_auc = [], []
for ds, auc, default in zip(model_24_10_result["dataset"],model_24_10_result["aucroc"], default_fomo_result['aucroc']):
    model_24_10_auc.append(auc)
    default_fomo_auc.append(default)

model_24_10_auc = np.array(model_24_10_auc)
default_fomo_auc = np.array(default_fomo_auc)

# Check differences
diffs = model_24_10_auc - default_fomo_auc
print("Mean improvement:", np.mean(diffs))
print("Better ratio:", np.mean(diffs > 0))

# One-sided Wilcoxon test
stat, p_value = wilcoxon(model_24_10_auc, default_fomo_auc, alternative='greater')

print("Statistic:", stat)
print("p-value:", p_value)

Mean improvement: 0.017096436544167835
Better ratio: 0.5785123966942148
Statistic: 4131.0
p-value: 0.12726800976074998


The model shows a small average improvement in AUC (+0.017) and outperforms the baseline on 58% of datasets. However, the difference is not statistically significant (Wilcoxon signed-rank test, p = 0.127)

In [34]:
# new files:
model_24_10_result = pd.read_csv('processed_model_24_10_results.csv')
default_fomo_result = pd.read_csv('processed_default_fomo.csv')

model_24_10_auc, default_fomo_auc = [], []
for ds, auc, default in zip(model_24_10_result["dataset"],model_24_10_result["aucroc"], default_fomo_result['aucroc']):
    model_24_10_auc.append(auc)
    default_fomo_auc.append(default)

model_24_10_auc = np.array(model_24_10_auc)
default_fomo_auc = np.array(default_fomo_auc)

# Check differences
diffs = model_24_10_auc - default_fomo_auc
print("Mean improvement:", np.mean(diffs))
print("Better ratio:", np.mean(diffs > 0))

# One-sided Wilcoxon test
stat, p_value = wilcoxon(model_24_10_auc, default_fomo_auc, alternative='greater')

print("Statistic:", stat)
print("p-value:", p_value)

Mean improvement: 0.040191282919340286
Better ratio: 0.6491228070175439
Statistic: 1060.0
p-value: 0.031781311612485603


### Compare p-value of test_validation_ensemble_13_5 with default_fomo

In [35]:
test_validation_ensemble_13_5_result = pd.read_csv('test_validation_ensemble_13_5.csv')

import numpy as np
from scipy.stats import wilcoxon
test_validation_ensemble_13_5_auc, default_fomo_auc = [], []
for ds, auc, default in zip(test_validation_ensemble_13_5_result["dataset"], test_validation_ensemble_13_5_result["aucroc"], default_fomo_result['aucroc']):
    test_validation_ensemble_13_5_auc.append(auc)
    default_fomo_auc.append(default)

test_validation_ensemble_13_5_auc = np.array(test_validation_ensemble_13_5_auc)
default_fomo_auc = np.array(default_fomo_auc)

# Check differences
diffs = test_validation_ensemble_13_5_auc - default_fomo_auc
print("Mean improvement:", np.mean(diffs))
print("Better ratio:", np.mean(diffs > 0))

# One-sided Wilcoxon test
stat, p_value = wilcoxon(test_validation_ensemble_13_5_auc, default_fomo_auc, alternative='greater')

print("Statistic:", stat)
print("p-value:", p_value)

Mean improvement: 0.05068799560347659
Better ratio: 1.0
Statistic: 1653.0
p-value: 2.1790595135160216e-14


### Compare p-value of test_validation_ensemble_13_5 with DTE

In [36]:
with open('competitor.json') as file:
    competitor = json.load(file)

In [37]:
test_validation_ensemble_13_5_result = pd.read_csv('test_validation_ensemble_13_5.csv')

import numpy as np
from scipy.stats import wilcoxon
test_validation_ensemble_13_5_auc, dte_auc = [], []

for ds, auc in zip(test_validation_ensemble_13_5_result["dataset"], test_validation_ensemble_13_5_result["aucroc"]):
    comp=competitor[ds+".npz"]
    test_validation_ensemble_13_5_auc.append(auc)
    dte_auc.append(comp)

test_validation_ensemble_13_5_auc = np.array(test_validation_ensemble_13_5_auc)
dte_auc = np.array(dte_auc)

# Check differences
diffs = test_validation_ensemble_13_5_auc - dte_auc
print("Mean improvement:", np.mean(diffs))
print("Better ratio:", np.mean(diffs > 0))

# One-sided Wilcoxon test
stat, p_value = wilcoxon(test_validation_ensemble_13_5_auc, dte_auc, alternative='greater')

print("Statistic:", stat)
print("p-value:", p_value)

Mean improvement: 0.01620309230835155
Better ratio: 0.49586776859504134
Statistic: 4013.0
p-value: 0.20208977384080434


## Compare model_24_10 with kNN_semi

In [63]:
#model_24_10_result = pd.read_csv('processed_model_24_10_results.csv')

# Load file 1
df1 = pd.read_csv("processed_model_24_10_results.csv")

# Load file 2
df2 = pd.read_csv("kNN_semi_AUCROC_all.csv").drop(columns=['KNN_5','KNN_10','KNN_20','KNN_40','KNN_50'])

# --- Clean KNN_avr column ---
df2["KNN_avr_clean"] = df2["KNN_avr"].str.split("(").str[0].astype(float) / 100.0

# --- Merge on dataset ---
import re

def clean_name(x):
    x = str(x)
    x = re.sub(r"^\d+_", "", x)  # remove prefix like 10_
    x = x.lower()                # lowercase everything
    x = x.strip()                # remove spaces
    return x
df1["dataset_clean"] = df1["dataset"].apply(clean_name)
df2["dataset_clean"] = df2["dataset"].apply(clean_name)
df = pd.merge(df1, df2, on="dataset_clean")

# Extract paired values
auc1 = np.array(df["aucroc"])
auc2 = np.array(df["KNN_avr_clean"])

# --- Compute metrics ---
diffs = auc1 - auc2

print("Mean improvement:", np.mean(diffs))
print("Better ratio:", np.mean(diffs > 0))

# --- Wilcoxon test (one-sided: is model1 better?) ---
stat, p_value = wilcoxon(auc1, auc2, alternative='greater')

print("Statistic:", stat)
print("p-value:", p_value)

Mean improvement: 0.0031067112664971
Better ratio: 0.42105263157894735
Statistic: 698.0
p-value: 0.8463631246570784


In [64]:
np.mean(df1['aucroc'])

0.7992558340735147

In [66]:
np.mean(df2["KNN_avr_clean"])

0.7961491228070177

In [65]:
df

,dataset_x,aucroc,aucpr,f1,dataset_clean,dataset_y,KNN_avr,KNN_avr_clean
0,10_cover,0.994912,0.774384,0.775862,cover,cover,96.73(0.22),0.9673
1,11_donors,0.999119,0.984495,0.982806,donors,donors,98.74(0.09),0.9874
2,12_fault,0.563585,0.584162,0.566122,fault,fault,60.22(0.00),0.6022
3,13_fraud,0.909840,0.046558,0.065217,fraud,fraud,95.57(0.93),0.9557
4,14_glass,0.994989,0.929268,0.900000,glass,glass,87.30(1.59),0.8730
5,15_Hepatitis,1.000000,1.000000,1.000000,hepatitis,hepatitis,87.90(1.75),0.8790
6,16_http,0.988390,0.383444,0.040816,http,http,99.96(0.01),0.9996
7,17_InternetAds,0.761846,0.614812,0.684783,internetads,internetads,65.73(0.00),0.6573
8,18_Ionosphere,0.937438,0.879533,0.911504,ionosphere,ionosphere,95.12(0.92),0.9512
9,19_landsat,0.610592,0.559162,0.497374,landsat,landsat,64.70(0.00),0.6470


### Merging datasets

In [16]:

# Function to map dataset → group name
def map_dataset(name):
    if name.startswith('MVTec-AD'):
        return 'MVTec-AD'
    elif name.startswith('SVHN'):
        return 'SVHN'
    elif name.startswith('20news'):
        return '20news'
    elif name.startswith('agnews'):
        return 'agnews'
    elif name.startswith('CIFAR10'):
        return 'CIFAR10'
    elif name.startswith('FashionMNIST'):
        return 'FashionMNIST'
    elif name.startswith('MNIST-C'):
        return 'MNIST-C'
    else:
        return None  # keep unchanged

df = pd.read_csv("model_24_10_result.csv", index_col=0)

metrics = ['aucroc', 'aucpr', 'f1']

# Apply mapping
df['group'] = df['dataset'].apply(map_dataset)

# Split
df_agg = df[df['group'].notna()].copy()
df_rest = df[df['group'].isna()].copy()

# Group and average
df_agg = df_agg.groupby('group', as_index=False)[metrics].mean()

# Rename back to dataset column
df_agg = df_agg.rename(columns={'group': 'dataset'})

# Combine
final_df = pd.concat([df_rest.drop(columns=['group']), df_agg], ignore_index=True)

# Sort
final_df = final_df.sort_values(by='dataset')

# Save WITHOUT index (important!)
final_df.to_csv("processedmodel_24_10_results.csv", index=False)

print("Saved to processed_results.csv")

Saved to processed_results.csv


In [17]:
def merging_datasets(df, file_name = 'processed_results.csv'):
    #df = pd.read_csv("model_24_10_result.csv", index_col=0)

    metrics = ['aucroc', 'aucpr', 'f1']

    # Apply mapping
    df['group'] = df['dataset'].apply(map_dataset)

    # Split
    df_agg = df[df['group'].notna()].copy()
    df_rest = df[df['group'].isna()].copy()

    # Group and average
    df_agg = df_agg.groupby('group', as_index=False)[metrics].mean()

    # Rename back to dataset column
    df_agg = df_agg.rename(columns={'group': 'dataset'})

    # Combine
    final_df = pd.concat([df_rest.drop(columns=['group']), df_agg], ignore_index=True)

    # Sort
    final_df = final_df.sort_values(by='dataset')

    # Save WITHOUT index (important!)
    final_df.to_csv(file_name, index=False)

    print(f"Saved to {file_name}")

In [18]:
default_fomo_result = pd.read_csv('default_fomo_result.csv', index_col=0)
merging_datasets(default_fomo_result, 'precessed_default_fomo.csv')

Saved to precessed_default_fomo.csv
